# Faruq-v3 ACMC — locked test amendment v2

Gunakan setelah audit v1 menghasilkan 129 parent bebas leakage tetapi FAIL hanya pada minimum support. Notebook ini tidak mengulang audit/download, tidak training, dan tidak tuning. Ia membekukan amendemen sebelum inference, lalu mengevaluasi D0FT–ACMC tiga seed sekali serta menjalankan paired parent bootstrap.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

REQUIRED = (
    'bundles/faruq-v3-locked-test-v1.tar',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0FT/D0FT_seed2026/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
LOCKED_ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
D0FT = tuple(require_project_artifact(PROJECT_ROOT, path) for path in (REQUIRED[2], REQUIRED[4], REQUIRED[6]))
ACMC = tuple(require_project_artifact(PROJECT_ROOT, path) for path in (REQUIRED[3], REQUIRED[5], REQUIRED[7]))
LOCKED_ROOT = Path('/content/faruq-v3-locked-test')
EVIDENCE_ROOT = PROJECT_ROOT / 'evidence/faruq-v3-locked-test-v1'
FINAL_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-locked-test-v2'
if not (LOCKED_ROOT / 'faruq_locked_test_eligibility.json').is_file():
    with tarfile.open(LOCKED_ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
ELIGIBILITY = LOCKED_ROOT / 'faruq_locked_test_eligibility.json'
AMENDMENT = EVIDENCE_ROOT / 'faruq_locked_test_amendment_v2.json'
assert ELIGIBILITY.is_file(), ELIGIBILITY
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
print('PROJECT:', PROJECT_ROOT)
print('TEST   :', LOCKED_ROOT)

In [ ]:
from coffee_detector.prepare_faruq_locked_test_amendment import prepare_faruq_locked_test_amendment
amendment = prepare_faruq_locked_test_amendment(ELIGIBILITY, AMENDMENT)
print(json.dumps(amendment, indent=2, ensure_ascii=False))
assert amendment['decision'] == 'PASS', 'STOP: amendemen v2 tidak lolos; jangan inference.'
assert amendment['model_inference_executed'] is False
print('PASS: protokol v2 beku sebelum inference.')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc_locked_test',
    '--test-root', str(LOCKED_ROOT),
    '--eligibility-summary', str(ELIGIBILITY),
    '--amendment-summary', str(AMENDMENT),
    '--confirmation-summary', str(CONFIRMATION),
    '--output-root', str(FINAL_ROOT),
    '--d0ft-checkpoints', *map(str, D0FT),
    '--acmc-checkpoints', *map(str, ACMC),
    '--seeds', '42', '123', '2026',
    '--device', '0', '--authorize-test',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = FINAL_ROOT / 'faruq_v3_acmc_locked_test_summary.json'
assert SUMMARY.is_file(), SUMMARY
final = json.loads(SUMMARY.read_text())
rows = [{'metric': metric, **values} for metric, values in final['aggregate'].items()]
display(pd.DataFrame(rows).style.format({key: '{:.2%}' for key in ('d0ft_mean', 'd0ft_std', 'acmc1_mean', 'acmc1_std', 'head_delta_mean', 'head_delta_std', 'head_delta_min')}))
print('BOOTSTRAP :', json.dumps(final['paired_parent_bootstrap'], indent=2))
print('CONCLUSION:', final['conclusion'])
print('CRITERIA  :', final['criteria'])
print('SUMMARY   :', SUMMARY)
print('Kirim tabel, bootstrap, dan conclusion. Test final sudah dibuka; tidak ada tuning berikutnya.')